1 全局向量初始化

In [7]:


MILVUS_URI="http://localhost:19530"
DB_NAME="rag_tutorial"
COLLECTION_NAME="docs"
KNOWLEDGE_FILE_TXT="../曲面打印机说明书.txt"
KNOWLEDGE_FILE_DOCS="../实施和定制开发服务工作说明书.docx"

EMBEDDING_MODEL_NAME="qwen3.7-text-embedding"
# EMBEDDING_MODEL_NAME="qwen3-vl-embedding"

EMBED_DIM=1024# qwen3.7-text-embedding的维度可选值：2560、2048、1536、1024、768、512、256


2 初始化Milvus

In [8]:
from pymilvus import  MilvusClient
#初始化客户端
milvus_client = MilvusClient(MILVUS_URI)
#查询已有数据库，如果没有就创建
existed_databases=milvus_client.list_databases()
if DB_NAME not in existed_databases:
    milvus_client.create_database(db_name=DB_NAME)
#切换到指定数据库
milvus_client.use_database(db_name=DB_NAME)
#创建collection,如果已存在指定名称的collection，为了避免冲突，需要将已有的删除，
if milvus_client.has_collection(collection_name=COLLECTION_NAME):
    milvus_client.drop_collection(collection_name=COLLECTION_NAME)
milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBED_DIM,
    metric_type="COSINE",
)

3初始化Embedding模型

In [9]:
from openai import OpenAI
import load_dotenv
import psycopg
import dashscope
import os
from dotenv import load_dotenv
load_dotenv(override=True)
my_api_key = os.getenv("ALL_API_KEY")
my_base_url = os.getenv("EMBEDDING_BASE_URL")
# 通用初始化
openai_client=OpenAI(
    api_key=my_api_key,
    base_url=my_base_url,
)



4 加载文档并切分

In [10]:
from langchain_community.document_loaders import TextLoader,UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#加载text文档
text_loader = TextLoader(
    file_path=KNOWLEDGE_FILE_TXT,   encoding="utf-8"
)
text_content=text_loader.load()
#加载DOCS文档
docs_loader=UnstructuredWordDocumentLoader(
    file_path=KNOWLEDGE_FILE_DOCS, encoding="utf-8"
)
docx_content=docs_loader.load()

#切分text文档
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    #曲面打印机说明书.txt文档切分定制化策略
    separators=[
        "\n\n",
        "\n",
        "\n★",
        "。",
        "；",
        "，",
        "、",
        "|",
        "：",
        ":",
        "-",
        "",
    ]
)
text_chunks=text_splitter.split_documents(text_content)
print(f"切分的chunk数量",len(text_chunks))
for i,chunk in enumerate(text_chunks):
    print(f"\n第{i}个:",chunk.page_content)



C:\Users\Administrator\AppData\Local\Temp\ipykernel_14276\2517892823.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader,UnstructuredWordDocumentLoader


切分的chunk数量 34

第0个: 安全指导
重要的安全指导 
请在使用您的打印机之前阅读以下指导，遵守标注在打印机上的警告和指导：
★不要堵塞该打印机外壳上的开口。
★不要将任何物品插入到打印机的槽中，注意不要将液体溅到打印机中。
★仅能使用220V 交流电源。
★避免所使用的机器与在开和关之间规则地切换的设备（如复印机或空调系统等）在同一回路中。
★避免使用由墙上开关或自动定时器控制的插座。
★将您的计算机系统远离潜在的电磁干扰源，例如：扬声器或无绳电话机座。
★不要使用损坏或磨损的电源线。
★不要试图自己修理打印机。
★在遇到下列情况时，请务必切断电源后请有经验的维修人员来维护：
如果液体溅入打印机中；
如果打印机摔下来或机壳破损；
如果打印机不能正常操作或在性能上有明显的变化.
1.2 使用打印机时注意事项
★不要在未切断电源的情况下用手移动打印头，否则可能会损坏打印机。
★始终使用电源开关来关闭打印机。该键被按下时，电源将被切断。在电源被切断之前，不要拔下打印机插头或数据线。
★在搬运打印机之前，应确保打印头在初始位置，并将其固定。
1.3 墨瓶使用注意事项

第1个: ★在搬运打印机之前，应确保打印头在初始位置，并将其固定。
1.3 墨瓶使用注意事项
★请将墨瓶置于小孩拿不到的地方，不要让小孩喝下或接触墨瓶。
★如果墨水粘到皮肤上，请用肥皂和水冲洗。如果墨水溅入眼中，请立即用水冲洗干净，并咨询医生。
★不要摇晃墨瓶，这会造成墨水泄漏。
★经常定时的清理废墨桶，避免废墨的溢出。
1.4 打印机安装位置
★将打印机放在水平、稳定的地面上。如果打印机倾斜，要先调整好机器的水平再进行下一步的安装。
★避免将打印机放置在温度和湿度容易发生剧烈变换的环境中。避免打印机受到阳光、强光或热源的直射。
★避免将打印机放置在容易振动或晃动的地方。
★在打印机周围留出足够的空间以保证打印机的正常工作。
1.5 工作环境
★保持室内温度范围在20—30℃之间，湿度范围在40—60%之间，要求配备空调与温湿度计。机器必须远离强干扰辐射源，地板必须平整。
1.6 电源供应
★支持AC 220V交流电源
★打印机必须良好接地（机壳和大地之间的电压不能大于0.3V,接地电阻应小于3Ω）。
★须配备UPS与电源稳压器。
安全指导
1.前言

第2个: ★须配备UPS与电源稳压器。
安

继续做DOCX文档的切分

In [11]:
docx_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    #实施和定制开发服务工作说明书.docx文档切分定制化策略
    separators = [
    "\n",
    "。",
    "；",
    "，",
    "、",
    "\t",
    "|",
    "：",
    ":",
    "-",
    ".",
    ")",
    "",
]
)
docx_chunks=docx_splitter.split_documents(docx_content)
print(f"切分的chunk数量",len(docx_chunks))
for i,chunk in enumerate(docx_chunks):
    print(f"\n第{i}个:",chunk.page_content)

切分的chunk数量 81

第0个: 今麦郎食品集团

集成供应链业务系统项目集

实施任务工作说明书(SOW)

文档说明

实施任务工作说明书(SOW，Statement of Work)主要说明上海科箭软件科技有限公司 (乙方)为所实施的今麦郎食品股份有限公司（甲方）的集成供应链项目集的如下内容：

项目背景，需求，范围描述

项目实施的软件产品功能描述

项目实施交付物定义

项目计划定义

项目职责定义

项目验收定义

运维服务定义

项目背景，需求，范围描述

项目背景

今麦郎集团于1994年成立，现年销售额超200亿元。其产品覆盖方便面、挂面、休闲食品、面粉、饮品等。经过28年的发展，今麦郎已经成为集研发、生产、销售于一体的现代化大型综合食品企业集团。公司产品远销美国、加拿大、澳大利亚、韩国、日本等海外36个国家和地区。年方便面生产能力120亿份，日处理小麦5500吨，年转化小麦180万吨的庞大生产规模，位居世界同行业前六名，国内入围前三强。

项目实施涉及的业务组织范围

项目实施范围

集团级OMS项目实施范围：今麦郎集团食品事业部、今麦郎集团饮品事业部

第1个: 项目实施范围

集团级OMS项目实施范围：今麦郎集团食品事业部、今麦郎集团饮品事业部

集团级WMS项目实施范围：今麦郎集团食品事业部方便面、休闲、挂面、三个标杆仓，以及兖州基地实施推广

今好运城配项目实施范围：今好运城配面品、饮品各一个标杆仓

实施地点：

集团级OMS和WMS实施地点：今麦郎隆尧总部基地和兖州基地

今好运城配实施地点：今好运指定的一个面品仓和一个饮品仓所在的城市

项目实现目标

本期项目的建设目标：

项目核心目标是构建集成供应链业务系统，提高生产及城配订单、仓储、物流管理运营效率。

建立集团级OMS，与集团干线TMS运输管理系统集成

与各事业部WMS集成（本次招标的项目集中包含食品WMS，饮品WMS已上线，后续需与集团级OMS对接）；

建立城配WMS及TMS运输管理系统并集成至集团级OMS，以支撑全国300个地级市DC仓和准DC“新四合一”业务发展。

本项目集分三期建设，大致规划如下：

一期（招标及报价范围，实施范围参考章节2.2）：

食品事业部OMS的建设

食品事业部各工厂成品库WMS建设（共计43个仓库）、

第2个: 食品事

5 生成向量并写入Milvus

In [31]:
from rich import print as rp
text_input=[
    chunk.page_content for chunk in text_chunks
]

docx_input=[
    chunk.page_content for chunk in docx_chunks
]
# #向量化过程

text_length=len(text_input)
docx_length=len(docx_input)
# 1. 将输入切分为每组10条（小于API限制的20条即可）
batch_size = 10
text_all_embeddings = []
docx_all_embeddings = []
# 2. 循环调用 API，每批次处理一部分文本
for i in range(0, text_length, batch_size):
    #设置每轮只取10条文本到batch里面
    batch = text_input[i : i + batch_size]

    rp(f"[bold green]正在处理第 {i//batch_size + 1} 批次... ([bold]{len(batch)} 条[/])")
    # print(len(batch))
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL_NAME,
        dimensions=EMBED_DIM,
        input=batch  # 这里只传入 10 条文本
    )

    # 3. 将当前批次的向量结果追加到总列表中
    batch_embeddings = [item.embedding for item in response.data]
    text_all_embeddings.extend(batch_embeddings)


for i in range(0, docx_length, batch_size):
    #设置每轮只取10条文本到batch里面
    batch = docx_input[i : i + batch_size]

    rp(f"[bold green]正在处理第 {i//batch_size + 1} 批次... ([bold]{len(batch)} 条[/])")
    # print(len(batch))
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL_NAME,
        dimensions=EMBED_DIM,
        input=batch  # 这里只传入 10 条文本
    )

    # 3. 将当前批次的向量结果追加到总列表中
    batch_embeddings = [item.embedding for item in response.data]
    docx_all_embeddings.extend(batch_embeddings)

#构建对应的数据
text_data=[
    {
            "id":i,
            "chunk_id":i,
            "vector":text_all_embeddings[i],
            "text":text_chunks[i].page_content,
            "source":KNOWLEDGE_FILE_TXT
    }
    for i in range(len(text_chunks))
]
docx_data=[
     {
            "id": len(text_chunks) + i,
            "chunk_id": len(text_chunks) + i,
            "vector":docx_all_embeddings[i],
            "text":docx_chunks[i].page_content,
            "source":KNOWLEDGE_FILE_DOCS
    }
    for i in range(len(docx_chunks))
]
text_insert_result=milvus_client.upsert(
    collection_name=COLLECTION_NAME,
    data=text_data,
)

docx_insert_result=milvus_client.upsert(
    collection_name=COLLECTION_NAME,
    data=docx_data,
)
print("text_insert_result:",text_insert_result)
print("docx_insert_result:",docx_insert_result)
#flush磁盘
milvus_client.flush(collection_name=COLLECTION_NAME)
#打印当前集合中的统计信息
stats=milvus_client.get_collection_stats(collection_name=COLLECTION_NAME)
print(stats)

正在处理第 1 批次... (10 条)

正在处理第 2 批次... (10 条)

正在处理第 3 批次... (10 条)

正在处理第 4 批次... (4 条)

正在处理第 1 批次... (10 条)

...

正在处理第 6 批次... (10 条)

正在处理第 7 批次... (10 条)

正在处理第 8 批次... (10 条)

正在处理第 9 批次... (1 条)

text_insert_result: {'upsert_count': 34, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]}
docx_insert_result: {'upsert_count': 81, 'ids': [34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114]}
{'row_count': 196}


In [30]:
#查询当前collections中有多少条数据
count_collections=milvus_client.query(
    collection_name=COLLECTION_NAME,
    filter="id>=0",
    output_fields=["id", "chunk_id"]
)
print(len(count_collections))

81


6 创建Agent

In [57]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化思考大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
)
myAgent=create_agent(
    model=myModel,
    # system_prompt="你是一个打印机操作助手，请根据检索到的上下文回答问题，如果上下文内容不足以回答，请直接回答：我不知道"
)

7 检索

In [53]:
#定义1个具体的函数，实现检索
def retrieve(user_query:str,limit:int=3):

    #将用户的问题向量化
    response=openai_client.embeddings.create(
        model="qwen3.7-text-embedding",
        dimensions=1024,
        input=user_query

    )
    #假设用户的问题只有1句话
    query_vector=response.data[0].embedding
    #从向量数据库中检索数据
    results=milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_vector],
        limit=limit,
        output_fields=["id", "chunk_id","text","source"]
    )
    return results[0]



8生产与回答问题

In [73]:
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage,ToolMessage
def generate_answer(user_query:str):
    #检索到的数据
    hits=retrieve(user_query)
    #格式化数据
    context_blocks=[]
    print("==============检索结果：==============")
    for i,hit in enumerate(hits):
        text=hit["entity"]["text"]
        source=hit["entity"]["source"]
        chunk_id=hit["entity"]["chunk_id"]
        score=hit["distance"]#Cosine模式下，score越高表示越相似
        print(f"[{i}] chunk_id={chunk_id}  source={source} score={score:.4f}")
        print(text)
        print()
        #拼接成带有编号和元数据的规范上下文块
        context_blocks.append(
            # f"[片段{i} | chunk_id={chunk_id} | source={source} \n {text}]"
              f"[参考片段{i} \n {text}]"
        )
    #将多个上下文片段用换行符连成1个大字符串
    context="\n\n".join(context_blocks)
    #构造1个agent需要用的Prompt

          # 【回答要求】：
            # 1. 必须只根据【已知信息】回答，不能凭空编造。
            # 2. 如果【已知信息】中完全没有包含回答【问题】所需的内容，请明确回答“根据当前已知信息，我无法回答该问题”。
            # 3. 回答语言请使用简体中文。
    #调用Agent
    system_prompt = f""""
            你是一个打印机操作助手。
            请忽略你自身已有的知识，**严格且仅仅根据下面的 <context> 标签内的文本内容**来回答用户的 <query>。

            <context>
            {context}
            </context>

            <query>
            {user_query}
            </query>

            【回答要求】
            1. 如果 <context> 标签内包含了回答 <query> 所需的确切信息，请直接给出答案。
            2. 如果 <context> 标签内完全没有提及 <query> 相关信息，请直接回答：“根据提供的已知信息，无法回答该问题。”，不要解释，不要道歉。
    """
    response=myAgent.invoke(
        {
            "messages":
             [
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_query)
            ]
        }
    )
    final_msg=response["messages"][-1]
    # #最终回答
    print("===============最终回答===============")
    final_msg.pretty_print()

测试

In [76]:
my_query="打印机最大负载的额定电流？"
generate_answer(my_query)

==============检索结果：==============
[0] chunk_id=8  source=../曲面打印机说明书.txt score=0.7240
|---:|:----|:-------|:--------|:----|:-------|:------------------------------------------------------------------|
|  0 |     | 功率(kw) | 交流电压(伏) | 相位  | 电流(安培) | 其他要求                                                              |
|  1 | 打印机 | 8.2    | 220±10% | 单相  | ＜20A   | 主机房内必须安装电器控制箱，包括：照明、打印机电源、空调、其他备用插座等。为了防止突发性的停电，设备必须联结到备用电源（发电机）。 |
4.打印机尺寸、包装尺寸、净重、毛重

[1] chunk_id=23  source=../曲面打印机说明书.txt score=0.7121
| 10 | 产品尺寸      | R50mm                                   |
| 11 | 干燥方式      | UV固化，即印即干                               |
| 12 | 操作系统      | 蒙泰 rip 软件，BYHX打印软件，win10 系统 64 操作系统位或以上 |
| 13 | 工作环境      | 20-30℃, 湿度 40-60%                       |
| 14 | 电源功率      | AC220V，1P+PE。50-60Hz，总功率 8.2kw          |
| 15 | 电柜尺寸      | 1000x700mm                              |
| 16 | 满载电流      | 19.5A                                   |
| 17 | 最大负载的额定电流 | 29.4A                                   |

[2] chu